# 01 – Explore a one-month TNP sample: column types

**Goal:** understand what the raw Chicago TNP trip export actually contains before writing the
`ingest` stage.

**Before running:** download subset of trips as CSV from the Chicago Data Portal and put the
file in `data/raw/sample`.

## Setup

The notebook runs from `notebooks/`, so relative paths like `configs/base.yaml` won't resolve.
We find the project root by walking up to `pyproject.toml` and build every path from there.
(If `load_config` is later changed to resolve paths itself, the `ROOT /` prefixes can go.)

In [ ]:
from pathlib import Path

import duckdb
import polars as pl

from rideforecast.config import load_config


def find_project_root(start: Path) -> Path:
    """Walk upward from `start` until a directory containing pyproject.toml is found."""
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError(f"No pyproject.toml found at or above {start}")


ROOT = find_project_root(Path.cwd())
cfg = load_config(ROOT / "configs" / "base.yaml")
RAW_DIR = ROOT / cfg.paths.raw / "sample"

files = sorted(RAW_DIR.glob("*.csv"))
assert files, f"No CSV files found in {RAW_DIR}"
for f in files:
    print(f"{f.name}: {f.stat().st_size / 1e9:.2f} GB")

pl.Config.set_tbl_rows(50)
pl.Config.set_fmt_str_lengths(60)

con = duckdb.connect()
SOURCE = (RAW_DIR / "*.csv").as_posix()

## 1. Look at the raw strings first

Reading every column as `VARCHAR` shows exactly what's in the file, with no type inference
in the way. All later checks run against this view, so we decide how to parse each column
rather than letting the reader decide.

In [ ]:
con.sql(f"""
    CREATE OR REPLACE VIEW raw_v AS
    SELECT * FROM read_csv('{SOURCE}', all_varchar = true)
""")

columns = con.sql("SELECT * FROM raw_v").columns
n_rows = con.sql("SELECT count(*) AS n FROM raw_v").pl().item()
print(f"{n_rows:,} rows, {len(columns)} columns")

con.sql("SELECT * FROM raw_v LIMIT 5").pl()

## 2. Can the inferred types be trusted?

By default DuckDB infers types from a sample of rows at the start of the file. A column that
looks like integers for the first 20k rows can contain a decimal or a stray string later on.
Comparing the sampled inference with a full-file scan shows which columns are risky.

The full scan reads the whole file, so this cell is slow on large files.

In [ ]:
def inferred_types(sample_size: int) -> pl.DataFrame:
    """Return the column types DuckDB infers when sniffing `sample_size` rows (-1 = all)."""
    rel = con.sql(f"DESCRIBE SELECT * FROM read_csv('{SOURCE}', sample_size = {sample_size})")
    return rel.pl().select("column_name", "column_type")


sampled = inferred_types(20_480)
full = inferred_types(-1)

type_comparison = sampled.join(full, on="column_name", suffix="_full_scan").with_columns(
    (pl.col("column_type") != pl.col("column_type_full_scan")).alias("differs")
)
type_comparison

## 3. Per-column summary

`SUMMARIZE` gives min, max, approximate distinct count and null percentage for every column
in one pass. Things to look for: numeric columns with impossible values (negative miles or
fares), columns that are almost entirely null, and ID-like columns whose distinct count
doesn't match the row count.

In [ ]:
summary = con.sql(f"SUMMARIZE SELECT * FROM read_csv('{SOURCE}', sample_size = -1)").pl()
summary.select("column_name", "column_type", "min", "max", "approx_unique", "null_percentage")

## 4. Timestamps

These are the most important columns for the project. Questions:

- What format are they stored in? Look at the raw strings before trusting any parse.
- Does one format parse every non-null value?
- Are they rounded to 15 minutes, as documented?
- Is there any timezone information in the string, or are they naive local times?

`TS_FORMAT` is a guess at the export format. If `failed_parse` is non-zero, look at the raw
examples and adjust it.

In [ ]:
TS_FORMAT = "%m/%d/%Y %I:%M:%S %p"
ts_cols = [c for c in columns if "timestamp" in c.lower()]

parse_rows = []
for col in ts_cols:
    parsed = f"try_strptime(\"{col}\", '{TS_FORMAT}')"
    row = con.sql(f"""
        SELECT
            '{col}' AS column_name,
            count("{col}") AS non_null,
            count("{col}") - count({parsed}) AS failed_parse,
            min({parsed}) AS earliest,
            max({parsed}) AS latest
        FROM raw_v
    """).pl()
    parse_rows.append(row)

pl.concat(parse_rows)

Check if any timestamps are not rounded to 15 minutes and to see whether there is an unequal distribution of start and end times.

In [ ]:
for col in ts_cols:
    parsed = f"try_strptime(\"{col}\", '{TS_FORMAT}')"
    print(f"\n{col}")
    print(
        con.sql(f"""
            SELECT minute({parsed}) AS minute, second({parsed}) AS second, count(*) AS n
            FROM raw_v
            WHERE {parsed} IS NOT NULL
            GROUP BY ALL
            ORDER BY n DESC
        """).pl()
    )

If there are trip end times that come before trip start times, then two things are possible:  
&emsp;1. There are errors in the trip time recording  
&emsp;2. Times are in local time

If only (2) is true, then there should be negative time trips in the 2023 Nov 5 sample but not in the 2026 sample.

In [ ]:
con.sql(f"""
    SELECT
        YEAR("Trip Start Timestamp") AS year,
        BOOL_OR("Trip End Timestamp" < "Trip Start Timestamp") AS has_negative_trips
    FROM read_csv('{SOURCE}')
    GROUP BY YEAR("Trip Start Timestamp")
    ORDER BY year;
""").pl()

In [ ]:
con.sql(f"""
    WITH b AS (
        SELECT "Trip Start Timestamp", 
            "Trip Start Timestamp"::DATE AS trip_date,
            HOUR("Trip Start Timestamp") AS trip_hour
        FROM read_csv('{SOURCE}', sample_size = -1)
    ) 
    SELECT trip_hour, COUNT(*)
    FROM b
    WHERE trip_date = '2024-03-10' 
    GROUP BY trip_hour
    ORDER BY trip_hour;
""").pl()

## 5. Community areas

The panel is built on pickup community area, so this column has to be clean. Chicago has 77
community areas numbered 1–77. Check how many values are null, how many don't parse as
integers, and how many fall outside 1–77.

In [ ]:
area_cols = [c for c in columns if "community area" in c.lower()]
print("Community area columns:", area_cols)

area_rows = []
for col in area_cols:
    as_int = f'TRY_CAST("{col}" AS INTEGER)'
    row = con.sql(f"""
        SELECT
            '{col}' AS column_name,
            COUNT(*) AS total,
            COUNT(*) - COUNT("{col}") AS null_values,
            COUNT("{col}") - COUNT({as_int}) AS not_integer,
            COUNT(*) FILTER (WHERE {as_int} NOT BETWEEN 1 AND 77) AS out_of_range,
            COUNT(DISTINCT {as_int}) AS distinct_areas
        FROM raw_v
    """).pl()
    area_rows.append(
        row.with_columns((pl.col("null_values") / pl.col("total") * 100).round(2).alias("null_pct"))
    )

pl.concat(area_rows)

In [ ]:
con.sql("""
    SELECT
        COUNT(*) FILTER (
            WHERE "Dropoff Centroid Location" IS NULL 
                AND "Dropoff Community Area" IS NULL
        )  / 
        COUNT(*) FILTER (
            WHERE "Dropoff Community Area" IS NULL
        ) AS missing_ca_dropoff,

        COUNT(*) FILTER (
            WHERE "Pickup Centroid Location" IS NULL 
                AND "Pickup Community Area" IS NULL
        ) / 
        COUNT(*) FILTER (
            WHERE "Pickup Community Area" IS NULL
        )AS missing_ca_pickup
    FROM raw_v
""").pl()

## 6. Dollar Columns

Making sure the dollar-valued columns are all numerical once '$' is removed.

In [ ]:
con.sql(f"""
    SELECT
        COUNT(*) FILTER (
            WHERE TRY_CAST(REPLACE("Fare", '$', '') AS DOUBLE) IS NULL
        ) AS fare_failures,

        COUNT(*) FILTER (
            WHERE TRY_CAST(REPLACE("Tip", '$', '') AS DOUBLE) IS NULL
        ) AS tip_failures,

        COUNT(*) FILTER (
            WHERE TRY_CAST(REPLACE("Additional Charges", '$', '') AS DOUBLE) IS NULL
        ) AS charge_failures,

        COUNT(*) FILTER (
            WHERE TRY_CAST(REPLACE("Trip Total", '$', '') AS DOUBLE) IS NULL
        ) AS total_failures
    FROM read_csv_auto('{SOURCE}');
""").pl()

In [ ]:
con.sql(f"""
    SELECT *
    FROM read_csv_auto('{SOURCE}')
    WHERE TRY_CAST(REPLACE("Trip Total", '$', '') AS DOUBLE) IS NULL
        AND "Trip Total" IS NOT NULL
    LIMIT 5;
""").pl()

In [ ]:
con.sql(f"""
    SELECT 
        COUNT(*) FILTER (
            "Trip Total" IS NULL
        ) AS total_missing_N,

        COUNT(*) FILTER (
            "Trip Total" IS NULL
                AND "Pickup Community Area" IS NULL
        ) AS pickup_missing_N,

        COUNT(*) FILTER (
            "Trip Total" IS NULL
                AND "Dropoff Community Area" IS NULL
        ) AS dropoff_missing_N
    FROM read_csv_auto('{SOURCE}')
    ;
""").pl()

## 7. Trip Seconds

In [ ]:
con.sql(f"""
    SELECT
        COUNT(*) FILTER (
            WHERE TRY_CAST(REPLACE("Trip Seconds", ',', '') AS DOUBLE) IS NULL
        ) AS trip_failures
    FROM read_csv_auto('{SOURCE}');
""").pl()

In [ ]:
con.sql(f"""
    SELECT *
    FROM read_csv_auto('{SOURCE}')
    WHERE TRY_CAST(REPLACE("Trip Seconds", ',', '') AS DOUBLE) IS NULL
        AND "Trip Seconds" IS NOT NULL
    LIMIT 5;
""").pl()

'999' is the max value for "Trip Seconds"

Need to check whether this because of ','

In [ ]:
con.sql(f"""
    WITH b AS (
        SELECT CAST(REPLACE("Trip Seconds", ',', '') AS DOUBLE) AS "Trip Seconds"
        FROM read_csv_auto('{SOURCE}')
    ) 
    SELECT MAX("Trip Seconds") as "max_trip_length"
    FROM b;
""").pl()

In [ ]:
con.sql(f"""
    SELECT
        COUNT(*) FILTER (
            WHERE "Trip Seconds" = '
        ) AS max_length,

        COUNT(*) FILTER (
            WHERE TRY_CAST(REPLACE("Tip", '$', '') AS DOUBLE) IS NULL
        ) AS zero_length
    FROM read_csv_auto('{SOURCE}')
    WHERE "Trip Seconds";
""").pl()

In [ ]:
con.sql(f"""
    SELECT MIN("Trip Start Timestamp") AS min_start, MAX("Trip Start Timestamp") AS max_start
    FROM read_csv_auto('{SOURCE}')
    WHERE YEAR("Trip Start Timestamp") = 2026;
""").pl()